In [ ]:
import os
from pathlib import Path

# Ensure relative paths resolve from the project root when running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import config
from klarity import parsing

## Process Images

Runs the full segmentation pipeline over  and writes one
Parquet file per replicate to .

Already-processed replicates are skipped automatically.

After this completes, run  to rebuild the
analysis DataFrames.

In [ ]:
import torch

model = parsing.load_yolo_model(config.MODEL_PATH)

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

model.to(device)
print(f"Model loaded  →  {config.MODEL_PATH.name}")
print(f"Device        →  {device}")

In [ ]:
parsing.process_all_settings(
    image_root_dir=config.IMAGE_DIR,
    model=model,
    output_dir=config.OUTPUT_DIR,
    conf=config.CONF,
    iou=config.IOU,
    binarize_thr=config.MASK_THR,
    overlays_root=str(config.OVERLAYS_DIR),
    save_masks_overlay=False,
    save_fit_overlay=True,
    geom_mode="hybrid",
)

## Next step

Rebuild the analysis DataFrames so plotting notebooks reflect the new data:

```bash
python scripts/build_dataframes.py
```